# SAM4Xtal mask statistics

Load a folder of annotations exported from the SAM4Xtal UI (`*.mask.json` + matching `*.mask.png`) and compute crystal / instance size statistics.

Each save produces:

| File | Contents |
| --- | --- |
| `<image>.mask.json` | Per-instance metadata: label, color, measurement (area, equiv. diameter, bbox), optional `nmPerPx` |
| `<image>.mask.png` | RGB colormap mask — black background; each instance painted with `instances[i].color` |

Downstream matching: keep PNG pixels whose RGB equals `instances[i].color` (or `hex`).

## 1. Configuration

Point `MASK_DIR` at the folder that holds your saved annotations. Relative paths are resolved from the notebook’s working directory (usually the repo root or `notebooks/`).

In [ ]:
from __future__ import annotations

from pathlib import Path

# --- edit this ---
MASK_DIR = Path("../masks")  # folder of *.mask.json + *.mask.png

# Optional: recompute area from the PNG (slower; useful as a sanity check)
VERIFY_PNG_PIXELS = True

# Optional: write summary CSVs next to the notebook
EXPORT_CSV = True
OUT_DIR = Path(".")  # where to write instance_stats.csv / image_stats.csv

MASK_DIR = MASK_DIR.expanduser().resolve()
print(f"MASK_DIR = {MASK_DIR}")
print(f"exists   = {MASK_DIR.is_dir()}")

In [ ]:
from collections import Counter
from typing import Any

import json
import math
import numpy as np
import pandas as pd
from PIL import Image

try:
    import matplotlib.pyplot as plt

    HAS_PLT = True
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed — plots will be skipped")

## 2. Load annotations

Discovers every `*.mask.json` under `MASK_DIR` (non-recursive by default). For each file, reads instance measurements from JSON and optionally recounts pixels from the matching PNG.

In [ ]:
def discover_mask_jsons(root: Path, recursive: bool = False) -> list[Path]:
    pattern = "**/*.mask.json" if recursive else "*.mask.json"
    return sorted(root.glob(pattern))


def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    h = hex_color.lstrip("#")
    if len(h) != 6:
        raise ValueError(f"expected #rrggbb, got {hex_color!r}")
    return int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)


def color_key(color: dict[str, Any] | None) -> tuple[int, int, int] | None:
    if not color:
        return None
    if all(k in color for k in ("r", "g", "b")):
        return int(color["r"]), int(color["g"]), int(color["b"])
    if "hex" in color and color["hex"]:
        return _hex_to_rgb(str(color["hex"]))
    return None


def count_instance_pixels(
    mask_png: Path,
    instances: list[dict[str, Any]],
) -> dict[str, int]:
    """Count pixels per instance id by matching RGB color in the colormap PNG."""
    img = np.asarray(Image.open(mask_png).convert("RGB"))
    # Flatten to (H*W, 3) for vectorized matching
    flat = img.reshape(-1, 3)
    counts: dict[str, int] = {}
    for inst in instances:
        iid = str(inst.get("id", inst.get("label", "")))
        rgb = color_key(inst.get("color"))
        if rgb is None:
            counts[iid] = 0
            continue
        counts[iid] = int(np.count_nonzero(np.all(flat == np.array(rgb), axis=1)))
    return counts


def load_annotation(
    json_path: Path,
    *,
    verify_png: bool = True,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """Return (meta, per-instance row dicts)."""
    with json_path.open(encoding="utf-8") as f:
        meta = json.load(f)

    mask_name = meta.get("maskFileName") or (
        json_path.name[: -len(".mask.json")] + ".mask.png"
    )
    mask_path = json_path.parent / Path(mask_name).name
    if not mask_path.is_file():
        alt = json_path.parent / (json_path.name[: -len(".mask.json")] + ".mask.png")
        if alt.is_file():
            mask_path = alt

    instances = meta.get("instances") or []
    png_counts: dict[str, int] | None = None
    png_total = None
    if verify_png and mask_path.is_file():
        png_counts = count_instance_pixels(mask_path, instances)
        png_total = sum(png_counts.values())
    elif verify_png:
        print(f"  warn: mask PNG missing for {json_path.name} (looked for {mask_path.name})")

    nm = meta.get("nmPerPx")
    image_w = meta.get("imageWidth")
    image_h = meta.get("imageHeight")
    image_area = (
        int(image_w) * int(image_h)
        if image_w is not None and image_h is not None
        else None
    )

    rows: list[dict[str, Any]] = []
    for inst in instances:
        m = inst.get("measurement") or {}
        iid = str(inst.get("id", inst.get("label", "")))
        area_px = m.get("areaPx")
        if area_px is None and png_counts is not None:
            area_px = png_counts.get(iid)

        # Re-derive nm fields if JSON lacks them but scale is known
        area_nm2 = m.get("areaNm2")
        equiv_d_px = m.get("equivDiameterPx")
        if equiv_d_px is None and area_px is not None and area_px > 0:
            equiv_d_px = 2.0 * math.sqrt(float(area_px) / math.pi)
        equiv_d_nm = m.get("equivDiameterNm")
        if nm is not None and area_px is not None:
            if area_nm2 is None:
                area_nm2 = float(area_px) * float(nm) * float(nm)
            if equiv_d_nm is None and equiv_d_px is not None:
                equiv_d_nm = float(equiv_d_px) * float(nm)

        rgb = color_key(inst.get("color"))
        row = {
            "source_json": json_path.name,
            "image_name": meta.get("imageName"),
            "image_id": meta.get("imageId"),
            "image_width": image_w,
            "image_height": image_h,
            "image_area_px": image_area,
            "nm_per_px": nm,
            "nm_per_px_source": meta.get("nmPerPxSource"),
            "saved_at": meta.get("savedAt"),
            "instance_id": iid,
            "label": inst.get("label"),
            "name": inst.get("name"),
            "color_hex": (inst.get("color") or {}).get("hex"),
            "color_rgb": rgb,
            "area_px": area_px,
            "area_px_png": png_counts.get(iid) if png_counts else None,
            "equiv_diameter_px": equiv_d_px,
            "bbox_width_px": m.get("bboxWidthPx"),
            "bbox_height_px": m.get("bboxHeightPx"),
            "area_nm2": area_nm2,
            "equiv_diameter_nm": equiv_d_nm,
            "bbox_width_nm": m.get("bboxWidthNm"),
            "bbox_height_nm": m.get("bboxHeightNm"),
            "confidence": m.get("confidence", inst.get("confidence")),
            "bbox_xyxy": inst.get("bbox_xyxy"),
            "n_points": len(inst.get("points") or []),
            "mask_png": mask_path.name if mask_path.is_file() else None,
            "mask_png_total_fg": png_total,
        }
        rows.append(row)
    return meta, rows


json_files = discover_mask_jsons(MASK_DIR, recursive=False)
if not json_files:
    # try one level of recursion if the folder is nested
    json_files = discover_mask_jsons(MASK_DIR, recursive=True)

print(f"Found {len(json_files)} annotation file(s)")
for p in json_files:
    try:
        print(f"  {p.relative_to(MASK_DIR)}")
    except ValueError:
        print(f"  {p}")

In [ ]:
all_rows: list[dict[str, Any]] = []
metas: list[dict[str, Any]] = []
load_errors: list[tuple[str, str]] = []

for jp in json_files:
    try:
        meta, rows = load_annotation(jp, verify_png=VERIFY_PNG_PIXELS)
        metas.append(meta)
        all_rows.extend(rows)
        print(f"  {jp.name}: {len(rows)} instance(s)")
    except Exception as e:
        load_errors.append((jp.name, str(e)))
        print(f"  ERROR {jp.name}: {e}")

df = pd.DataFrame(all_rows)
print(f"\nTotal instances: {len(df)}")
if load_errors:
    print(f"Errors: {len(load_errors)}")

if df.empty:
    raise SystemExit(
        f"No instances loaded from {MASK_DIR}. "
        "Export annotations from the UI (Save annotation) into this folder."
    )

df.head()

## 3. Per-image summary

Instance counts, total / mean crystal area, and image coverage fraction.

In [ ]:
def _first(s: pd.Series):
    return s.iloc[0] if len(s) else None


agg = {
    "instance_id": "count",
    "area_px": ["sum", "mean", "median", "std", "min", "max"],
    "equiv_diameter_px": ["mean", "median", "std", "min", "max"],
    "image_width": "first",
    "image_height": "first",
    "image_area_px": "first",
    "nm_per_px": "first",
    "nm_per_px_source": "first",
    "image_name": "first",
    "saved_at": "first",
}

# Only include nm columns when present
if df["area_nm2"].notna().any():
    agg["area_nm2"] = ["sum", "mean", "median", "std", "min", "max"]
if df["equiv_diameter_nm"].notna().any():
    agg["equiv_diameter_nm"] = ["mean", "median", "std", "min", "max"]

by_image = df.groupby("source_json", dropna=False).agg(agg)
by_image.columns = [
    "_".join(c).strip("_") if isinstance(c, tuple) else c for c in by_image.columns
]
by_image = by_image.rename(
    columns={
        "instance_id_count": "n_instances",
        "image_width_first": "image_width",
        "image_height_first": "image_height",
        "image_area_px_first": "image_area_px",
        "nm_per_px_first": "nm_per_px",
        "nm_per_px_source_first": "nm_per_px_source",
        "image_name_first": "image_name",
        "saved_at_first": "saved_at",
        "area_px_sum": "total_area_px",
        "area_px_mean": "mean_area_px",
        "area_px_median": "median_area_px",
        "area_px_std": "std_area_px",
        "area_px_min": "min_area_px",
        "area_px_max": "max_area_px",
    }
)

if "image_area_px" in by_image.columns:
    by_image["coverage_frac"] = by_image["total_area_px"] / by_image["image_area_px"]
    by_image["coverage_pct"] = 100.0 * by_image["coverage_frac"]

by_image = by_image.reset_index()
display_cols = [
    c
    for c in [
        "source_json",
        "image_name",
        "n_instances",
        "total_area_px",
        "mean_area_px",
        "median_area_px",
        "coverage_pct",
        "nm_per_px",
        "area_nm2_sum",
        "equiv_diameter_nm_mean",
        "equiv_diameter_nm_median",
    ]
    if c in by_image.columns
]
by_image[display_cols]

## 4. Global instance statistics

Pooled across every crystal in the folder. Prefer **nm** columns when `nmPerPx` was set at save time; otherwise use pixel units.

In [ ]:
METRIC_COLS = [
    "area_px",
    "equiv_diameter_px",
    "bbox_width_px",
    "bbox_height_px",
    "area_nm2",
    "equiv_diameter_nm",
    "bbox_width_nm",
    "bbox_height_nm",
    "confidence",
]


def describe_metrics(frame: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        if col not in frame.columns:
            continue
        s = pd.to_numeric(frame[col], errors="coerce").dropna()
        if s.empty:
            continue
        rows.append(
            {
                "metric": col,
                "n": int(s.count()),
                "mean": float(s.mean()),
                "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
                "min": float(s.min()),
                "p25": float(s.quantile(0.25)),
                "median": float(s.median()),
                "p75": float(s.quantile(0.75)),
                "p90": float(s.quantile(0.90)),
                "max": float(s.max()),
            }
        )
    return pd.DataFrame(rows).set_index("metric")


stats = describe_metrics(df, METRIC_COLS)
stats

In [ ]:
# High-level rollup
n_images = df["source_json"].nunique()
n_inst = len(df)
with_scale = int(df["nm_per_px"].notna().sum())
total_area_px = pd.to_numeric(df["area_px"], errors="coerce").sum()

print("=== Folder summary ===")
print(f"Images (annotations): {n_images}")
print(f"Instances (crystals): {n_inst}")
print(f"Instances with nm scale: {with_scale} / {n_inst}")
print(f"Total masked area: {total_area_px:,.0f} px²")
print(f"Mean instances / image: {n_inst / max(n_images, 1):.2f}")

if df["equiv_diameter_nm"].notna().any():
    d = pd.to_numeric(df["equiv_diameter_nm"], errors="coerce").dropna()
    print(
        f"Equiv. diameter (nm): mean={d.mean():.2f}, median={d.median():.2f}, "
        f"std={d.std(ddof=1) if len(d) > 1 else 0:.2f}, n={len(d)}"
    )
else:
    d = pd.to_numeric(df["equiv_diameter_px"], errors="coerce").dropna()
    print(
        f"Equiv. diameter (px): mean={d.mean():.2f}, median={d.median():.2f}, "
        f"std={d.std(ddof=1) if len(d) > 1 else 0:.2f}, n={len(d)}"
    )

if df["area_nm2"].notna().any():
    a = pd.to_numeric(df["area_nm2"], errors="coerce").dropna()
    print(
        f"Area (nm²): mean={a.mean():.2f}, median={a.median():.2f}, n={len(a)}"
    )
else:
    a = pd.to_numeric(df["area_px"], errors="coerce").dropna()
    print(f"Area (px²): mean={a.mean():.2f}, median={a.median():.2f}, n={len(a)}")

## 5. PNG vs JSON area check

When `VERIFY_PNG_PIXELS` is on, compare JSON `measurement.areaPx` to a recount of colormap pixels. Small differences can come from polygon rasterization vs. the stored prediction area.

In [ ]:
png_check = None
if VERIFY_PNG_PIXELS and "area_px_png" in df.columns and df["area_px_png"].notna().any():
    check = df[["source_json", "label", "name", "area_px", "area_px_png"]].copy()
    check["delta_px"] = (
        pd.to_numeric(check["area_px_png"], errors="coerce")
        - pd.to_numeric(check["area_px"], errors="coerce")
    )
    check["rel_err"] = check["delta_px"] / pd.to_numeric(
        check["area_px"], errors="coerce"
    ).replace(0, np.nan)
    print(
        f"PNG vs JSON |delta| mean={check['delta_px'].abs().mean():.2f} px, "
        f"max={check['delta_px'].abs().max():.0f} px"
    )
    # Largest mismatches first
    png_check = check.reindex(
        check["delta_px"].abs().sort_values(ascending=False).index
    ).head(15)
else:
    print("PNG verification skipped or no PNG areas available.")

png_check

## 6. Distributions

In [ ]:
if not HAS_PLT:
    print("Install matplotlib to plot: pip install matplotlib")
else:
    use_nm = df["equiv_diameter_nm"].notna().any()
    diam_col = "equiv_diameter_nm" if use_nm else "equiv_diameter_px"
    area_col = "area_nm2" if (use_nm and df["area_nm2"].notna().any()) else "area_px"
    diam_unit = "nm" if diam_col.endswith("_nm") else "px"
    area_unit = "nm²" if area_col.endswith("_nm2") else "px²"

    diam = pd.to_numeric(df[diam_col], errors="coerce").dropna()
    area = pd.to_numeric(df[area_col], errors="coerce").dropna()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].hist(diam, bins=min(30, max(5, int(np.sqrt(len(diam))))), edgecolor="black", alpha=0.8)
    axes[0].set_xlabel(f"Equivalent diameter ({diam_unit})")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Size distribution")
    axes[0].axvline(diam.median(), color="C1", ls="--", label=f"median={diam.median():.1f}")
    axes[0].axvline(diam.mean(), color="C3", ls=":", label=f"mean={diam.mean():.1f}")
    axes[0].legend(fontsize=8)

    axes[1].hist(area, bins=min(30, max(5, int(np.sqrt(len(area))))), edgecolor="black", alpha=0.8, color="C2")
    axes[1].set_xlabel(f"Area ({area_unit})")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Area distribution")

    counts = df.groupby("source_json").size()
    axes[2].bar(range(len(counts)), counts.values, color="C4", edgecolor="black")
    axes[2].set_xticks(range(len(counts)))
    axes[2].set_xticklabels(
        [Path(s).stem.replace(".mask", "") for s in counts.index],
        rotation=45,
        ha="right",
        fontsize=8,
    )
    axes[2].set_ylabel("Instances")
    axes[2].set_title("Instances per image")

    fig.tight_layout()
    plt.show()

    # Box / strip of diameter by image when multiple images
    if df["source_json"].nunique() > 1:
        fig2, ax = plt.subplots(figsize=(max(6, 1.2 * df["source_json"].nunique()), 4))
        labels = []
        data = []
        for src, g in df.groupby("source_json"):
            s = pd.to_numeric(g[diam_col], errors="coerce").dropna()
            if s.empty:
                continue
            data.append(s.values)
            labels.append(Path(str(src)).stem.replace(".mask", ""))
        if data:
            ax.boxplot(data, labels=labels)
            ax.set_ylabel(f"Equivalent diameter ({diam_unit})")
            ax.set_title("Diameter by image")
            plt.xticks(rotation=45, ha="right")
            fig2.tight_layout()
            plt.show()

## 7. Full instance table & export

In [ ]:
table_cols = [
    c
    for c in [
        "source_json",
        "image_name",
        "label",
        "name",
        "area_px",
        "area_px_png",
        "equiv_diameter_px",
        "bbox_width_px",
        "bbox_height_px",
        "area_nm2",
        "equiv_diameter_nm",
        "bbox_width_nm",
        "bbox_height_nm",
        "confidence",
        "nm_per_px",
        "color_hex",
    ]
    if c in df.columns
]

instance_table = df[table_cols].sort_values(
    ["source_json", "label"], kind="stable"
).reset_index(drop=True)
instance_table

In [ ]:
if EXPORT_CSV:
    OUT_DIR = OUT_DIR.expanduser().resolve()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    inst_path = OUT_DIR / "instance_stats.csv"
    img_path = OUT_DIR / "image_stats.csv"
    summary_path = OUT_DIR / "metric_summary.csv"

    instance_table.to_csv(inst_path, index=False)
    by_image.to_csv(img_path, index=False)
    stats.to_csv(summary_path)

    print(f"Wrote {inst_path}")
    print(f"Wrote {img_path}")
    print(f"Wrote {summary_path}")
else:
    print("EXPORT_CSV is False — skipping CSV write.")

## 8. Optional: preview one mask

Quick visual of the first annotation’s RGB colormap PNG (black = background).

In [ ]:
if not HAS_PLT:
    print("matplotlib required for preview")
else:
    first = json_files[0]
    stem = first.name[: -len(".mask.json")]
    png = first.parent / f"{stem}.mask.png"
    if not png.is_file():
        # try maskFileName from json
        with first.open(encoding="utf-8") as f:
            m = json.load(f)
        candidate = first.parent / Path(m.get("maskFileName", "")).name
        png = candidate if candidate.is_file() else png

    if png.is_file():
        arr = np.asarray(Image.open(png).convert("RGB"))
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(arr)
        ax.set_title(png.name)
        ax.axis("off")
        # legend swatches from df for this file
        sub = df[df["source_json"] == first.name]
        handles = []
        labels = []
        for _, r in sub.iterrows():
            hex_c = r.get("color_hex") or "#ffffff"
            handles.append(plt.matplotlib.patches.Patch(facecolor=hex_c, edgecolor="k"))
            labels.append(f"{r.get('label')}: {r.get('name')} ({r.get('area_px')} px²)")
        if handles:
            ax.legend(handles, labels, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
        fig.tight_layout()
        plt.show()
        fg = np.any(arr != 0, axis=2).sum()
        print(f"{png.name}: shape={arr.shape}, foreground_pixels={fg}")
    else:
        print(f"No mask PNG found next to {first.name}")

---

### Dependencies

```bash
pip install numpy pandas pillow matplotlib jupyter
```

### Typical workflow

1. In the SAM4Xtal UI, segment crystals and click **Save annotation** for each image.
2. Collect the downloaded `<name>.mask.json` / `<name>.mask.png` pairs into one folder.
3. Set `MASK_DIR` above to that folder and run all cells.
4. Use `instance_stats.csv` for further analysis (Excel, R, etc.).